In [20]:
!pip install torch transformers pandas regex faiss-cpu numpy gradio sentence_transformers rouge-score bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.7 MB/s eta 0:00:00


In [21]:
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import torch
import gradio as gr
from sklearn.preprocessing import normalize

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import score as bert_score

In [5]:
# Configuration
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"
CHUNK_SIZE = 100
MIN_SIMILARITY = 0.7

# Preprocessing
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data = pd.read_csv("/Parts_processed.csv")
data = data[["ID", "DESCRIPTION"]].dropna()
data['clean_desc'] = data['DESCRIPTION'].apply(preprocess_text)

# Document chunks
documents = []
metadata = []
for _, row in data.iterrows():
    text = f"ID {row['ID']}: {row['clean_desc']}"
    # Simple chunking if descriptions are long
    for i in range(0, len(text), CHUNK_SIZE):
        chunk = text[i:i+CHUNK_SIZE]
        documents.append(chunk)
        metadata.append({
            "id": row['ID'],
            "full_desc": row['DESCRIPTION'],
            "chunk_num": i//CHUNK_SIZE
        })

# Embeddings with normalization
embedder = SentenceTransformer(EMBEDDING_MODEL)
doc_embeddings = embedder.encode(documents, show_progress_bar=True)
doc_embeddings = normalize(doc_embeddings).astype('float32')

# FAISS index - cosine similarity
index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

# Document ID
id_to_docs = data.set_index('ID')['DESCRIPTION'].to_dict()

# LLM setup
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=0.7,
    max_new_tokens=256
)

def find_similar_parts(query_id_or_text, k=5):
    # Check if input is a part ID
    if str(query_id_or_text).isdigit() and int(query_id_or_text) in id_to_docs:
        query_text = preprocess_text(id_to_docs[int(query_id_or_text)])
    else:
        query_text = preprocess_text(query_id_or_text)

    # Embed query
    query_embedding = embedder.encode([query_text])
    query_embedding = normalize(query_embedding).astype('float32')

    # Search with score threshold
    scores, indices = index.search(query_embedding, k*2)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if score >= MIN_SIMILARITY:
            results.append({
                "id": metadata[idx]["id"],
                "score": score,
                "description": metadata[idx]["full_desc"]
            })

    # Deduplicate and sort
    seen = set()
    final_results = []
    for r in sorted(results, key=lambda x: -x['score']):
        if r['id'] not in seen:
            seen.add(r['id'])
            final_results.append(r)
            if len(final_results) >= k:
                break

    return final_results[:k]

def generate_answer(query):
    similar_parts = find_similar_parts(query)

    if not similar_parts:
        return "No similar parts found based on current data."

    # Build context string
    context = "\n".join(
        f"Part ID {p['id']} (similarity: {p['score']:.2f}): {p['description']}"
        for p in similar_parts
    )

    # Enhanced prompt structure
    prompt = f"""<|system|>
    You're a technical parts assistant. List similar parts based on this context:
    {context}

    <|user|>
    Show similar parts for: {query}

    <|assistant|>
    """

    # Generate response
    response = generator(
        prompt,
        max_new_tokens=512,
        do_sample=True,
        num_beams=3
    )

    return response[0]['generated_text'].split("<|assistant|>")[-1].strip()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

Device set to use cuda:0


In [8]:
# Gradio Code
def main():
  demo = gr.Interface(
      fn=generate_answer,
      inputs=gr.Textbox(label="Enter Part ID or Description"),
      outputs=gr.Textbox(label="Similar Parts"),
      title="Technical Parts Finder",
      description="Enter a Part ID or description to find similar components",
      examples=[["A1"], ["Attribute: VERY FAST"]]
  )

  demo.launch()

In [9]:
main()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5875c9fc4f0f29d337.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
main()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dd7e1d639b692a1369.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [26]:
# Evaluation data (text-to-text comparison)
evaluation_data = [
    {
        "query": "give me similar IDs of A1 based on description",
        "reference_answer": "Indicator Red Fast Movement 1.6A 250V Holder Plastic 5 X 20mm Ceramic Box CCC/PSE/VDE/cULus Electric Indicator, Very Fast Blow, 1.6A, 250VAC, 1500A (IR), Inline/holder, 5x20mm"
    },
    {
        "query": "give Red Fast Movement 1.6A 250V Holder Plastic IDs",
        "reference_answer": "Indicator Red Fast Movement 1.6A 250V Holder Plastic 5 X 20mm Ceramic Bulk CCC/CE/CSA/KC/PSE/SEMKO/UL/VDE Electric Indicator, Fast Blow, 1.6A, 250VAC, 1500A (IR), Inline/holder, 5x20mm"
    }
]

def evaluate_generation(evaluation_data):
    bleu_scores = []
    rouge_l_scores = []
    meteor_scores = []
    bert_f1_scores = []
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    smoothing_function = SmoothingFunction().method1

    for example in evaluation_data:
        query = example["query"]
        reference = example["reference_answer"]
        generated = generate_answer(query)

        # BLEU
        bleu = sentence_bleu([reference.split()], generated.split(), smoothing_function=smoothing_function)
        bleu_scores.append(bleu)

        # ROUGE-L
        rouge_scores = scorer.score(reference, generated)
        rouge_l_scores.append(rouge_scores['rougeL'].fmeasure)


        # BERTScore (F1)
        P, R, F1 = bert_score([generated], [reference], lang="en", verbose=False)
        bert_f1_scores.append(F1[0].item())

    avg_bleu = sum(bleu_scores) / len(bleu_scores)
    avg_rouge_l = sum(rouge_l_scores) / len(rouge_l_scores)
    avg_bert_f1 = sum(bert_f1_scores) / len(bert_f1_scores)

    return avg_bleu, avg_rouge_l, avg_bert_f1

avg_bleu, avg_rouge_l, avg_bert_f1 = evaluate_generation(evaluation_data)
print(f"\ Evaluation Metrics:")
print(f" BLEU: {avg_bleu:.2f}")
print(f" ROUGE-L: {avg_rouge_l:.2f}")
print(f" BERTScore (F1): {avg_bert_f1:.2f}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Generation Evaluation Metrics:
 BLEU: 0.06
 ROUGE-L: 0.17
 BERTScore (F1): 0.82
